# 091 — Síntesis de voz y derechos de identidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** (a) 5 × 24 000 = **120 000 muestras**. (b) 120 000 / 300 = **400
frames** (división exacta; con padding de borde podría ser 401). (c) 400 × 80 =
**32 000 valores**. (d) 120 000 / 32 000 = **3.75×** — y además el mel descarta la
fase, así que la compresión de información es mayor que la de tamaño.

**Ejercicio 2.** Autorregresivo: 120 000 / 1 000 = 120 s de cómputo → RTF = 120/5 =
**24** (24× más lento que tiempo real). Paralelo: RTF = 0.08/5 = **0.016**. Razón =
24 / 0.016 = **1 500×**. Solo el paralelo (RTF < 1) sirve para conversación en vivo;
la diferencia es estructural: 120 000 pasos secuenciales vs unas decenas de capas.

**Ejercicio 3.** (a) consentimiento de uso de datos para entrenamiento; (b) permiso
específico de clonación de identidad vocal (generar habla que la persona nunca dijo);
(c) cesión de derechos de imagen/voz con fines comerciales. La clonación zero-shot
sin permiso viola (b) — y (c) si se explota comercialmente — aunque el podcast fuera
público: publicar una grabación no cede el timbre como identidad reutilizable.

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; solo la evidencia
inspeccionable autoriza conclusiones.


In [ ]:
result = run_lab("safety", seed=91)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica de los ejercicios
# Ejercicio 1: dimensiones del mel
duracion, sr, hop, n_mels = 5, 24_000, 300, 80
muestras = duracion * sr
frames = muestras // hop
valores_mel = frames * n_mels
print(f"muestras={muestras}  frames={frames}  mel={frames}x{n_mels}={valores_mel}")
print(f"compresión = {muestras / valores_mel:.2f}x")
assert (muestras, frames, valores_mel) == (120_000, 400, 32_000)

# Ejercicio 2: RTF autorregresivo vs paralelo
computo_ar = muestras / 1_000          # 1000 muestras por segundo de cómputo
rtf_ar = computo_ar / duracion
rtf_par = 0.08 / duracion
print(f"RTF autorregresivo = {rtf_ar:.1f}   RTF paralelo = {rtf_par:.3f}")
print(f"razón = {rtf_ar / rtf_par:.0f}x")
assert rtf_ar == 24.0 and abs(rtf_par - 0.016) < 1e-9


## Reflexión

1. El mel-espectrograma descarta la fase: ¿por qué eso obliga a que el vocoder sea un
   modelo generativo y qué pasaría si reconstruyeras la onda con fase cero?
2. Si SV2TTS clona un timbre con segundos de audio público, ¿qué control de
   consentimiento es técnicamente viable: restringir los datos, marcar el audio
   sintético (watermarking/C2PA), o verificar identidad antes de clonar? Justifica.
3. FastSpeech 2 elimina los fallos de atención de Tacotron 2 usando un predictor de
   duración explícito: ¿qué gana y qué pierde en prosodia frente al modelo
   autorregresivo?
